# Lab 07-04 — Golden-set regression: catch a pipeline regression before it ships

**Track 07 · Evaluation** — a golden set is a small, hand-checked set of (question, reference) pairs that never changes — the evaluation anchor. The repo's is `src/evaluation/golden.py`: 23 verified invoice QA pairs (source of truth from the SD-08 notebooks). This lab turns the sample-invoice + Invoice_1 pairs into a regression gate.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, faiss, pypdf, and Groq directly — no repo component library. Every block of the pipeline is built right here: the PDF loader, the BGE embedder, the FAISS index, the top-k retriever, the answer generator, the faithfulness judge, and the reference-based correctness helpers — all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

Two variants of the same pipeline, differing in exactly one knob:

* **Variant A — the reference pipeline**: full PDF -> pages -> BGE embeddings -> FAISS -> top-3 retrieval -> Groq answer. Scored on faithfulness (claims supported by context) and reference correctness (gold contained in the answer, answer-reference cosine).
* **Variant B — a REGRESSED pipeline**: the same code with `top_k=1` (one context chunk instead of three) — a realistic silent degradation (someone "optimized" the retriever).

The gate asserts A >= B on every metric. That is the regression contract: any future change that drops a metric below the reference pipeline FAILS the gate, no matter how good the new code looks. A golden set that never moves is what makes the comparison fair.

The pipeline, drawn inline:

```text
golden questions (8, from evaluation/golden.py)
  -> one index over the invoice PDFs (PyPDFLoader -> BGE -> FAISS)
  -> variant A: top_k = 3   (reference)
  -> variant B: top_k = 1   (the would-be regression)
  -> score faithfulness / contained / cosine per variant
  -> regression gate: A >= B (contained, cosine strict; faithfulness tolerated)
```


## Setup

One prerequisite must hold before this notebook will run:

- **SD-08 invoice PDFs on disk** — `Data/SD-08-invoices/` (`sample-invoice.pdf` + `Invoice_1.pdf`), already fetched by the repo's manifest-verified fetchers.
- **`GROQ_API_KEY` in the repo-root `.env`** — the generator and the faithfulness judge both run on Groq's hosted Llama model; the imports cell loads the key via python-dotenv.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-groq`, `sentence-transformers`, `faiss-cpu`, `pypdf`, `python-dotenv`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`):
#   sentence-transformers -> local BGE embeddings (HuggingFaceEmbeddings)
#   langchain-huggingface -> the HuggingFaceEmbeddings wrapper
#   langchain-community   -> the FAISS vector store + PyPDFLoader
#   faiss-cpu             -> the FAISS index
#   pypdf                 -> PDF text extraction (PyPDFLoader backend)
#   langchain-groq        -> ChatGroq (generator + judge)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu pypdf langchain-groq python-dotenv


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

# LangChain + sentence-transformers + faiss + pypdf + Groq — the only
# libraries this notebook needs. Nothing is imported from the repo's src/
# component library.
from dotenv import load_dotenv  # noqa: E402
from langchain_community.document_loaders import PyPDFLoader  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)
load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration — the golden set and the two pipeline variants

`GOLDEN_QA` (from `src/evaluation/golden.py`) fixes the contract: 8 questions with substantive gold answers over the SD-08 invoice PDFs. Variant **A** retrieves `top_k = 3` context chunks; variant **B** retrieves `top_k = 1`. Everything else — index, questions, references, generator, judge — is shared, so any score delta is attributable to that one knob. `FAITHFULNESS_TOLERANCE = 0.15` gives the judge-based metric room to breathe: faithfulness is claim-support DENSITY, not completeness — a pipeline that answers less can score higher (fewer claims, fewer chances to hallucinate).


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — the golden set and the two pipeline variants
# --------------------------------------------------------------------------
INVOICE_DIR = Path("Data/SD-08-invoices")
GOLDEN_DOCS = ["sample-invoice", "Invoice_1"]  # subset of GOLDEN_QA keys
VARIANT_A_TOP_K = 3  # reference pipeline
VARIANT_B_TOP_K = 1  # the regression: single-chunk context
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"  # local embedder (cached)
GROQ_MODEL_NAME = "llama-3.3-70b-versatile"  # generator + judge (lab default)

# Faithfulness is judge-based and measures claim-support DENSITY, not answer
# completeness: a pipeline that answers less can score HIGHER (fewer claims,
# fewer chances to hallucinate). It is also noisy on a 10-question sample.
# So the regression gate tolerates a small faithfulness dip while gating the
# reference-anchored metrics (containment, cosine) strictly.
FAITHFULNESS_TOLERANCE = 0.15

# The golden set, verbatim from src/evaluation/golden.py (the two docs this
# lab gates on). Hand-checked invoice QA pairs — the evaluation anchor.
GOLDEN_QA: dict[str, list[tuple[str, str]]] = {
    "sample-invoice": [
        ("What is the invoice number?", "INV-100"),
        ("What is the total due on the invoice?", "610.00"),
        ("Who is the customer on this invoice?", "MICROSOFT CORPORATION"),
        ("What was the sales tax amount?", "10.00"),
    ],
    "Invoice_1": [
        ("What is the invoice number?", "3847193"),
        ("What is the total price of all items?", "1075.70"),
        ("How many pieces were delivered in total?", "66"),
        ("Which item code is the bubble film roll?", "JF9912413BF"),
    ],
}


## 2. Reference-based correctness (same helpers as labs 02-03)

The deterministic scoring block reused verbatim: `normalize` (lowercase, strip punctuation/articles), `reference_contained` (the normalized gold answer appears inside the normalized answer — the exact-style check that survives elaboration), and `cosine_similarity` (semantic tolerance for rephrasing). One pipeline, two scoring philosophies.


In [ ]:
# --------------------------------------------------------------------------
# 2. Reference-based correctness (same helpers as labs 02-03)
# --------------------------------------------------------------------------
def normalize(text: str) -> str:
    """Lowercase, strip punctuation/articles and whitespace."""
    cleaned = "".join(c.lower() for c in text if c.isalnum() or c.isspace())
    words = [w for w in cleaned.split() if w not in ("a", "an", "the")]
    return " ".join(words)


def reference_contained(answer: str, reference: str) -> bool:
    """True when the normalized gold answer appears inside the answer."""
    return normalize(reference) in normalize(answer)


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors (0.0 if either is zero)."""
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(x * x for x in b) ** 0.5
    if na == 0.0 or nb == 0.0:
        return 0.0
    return dot / (na * nb)


## 3. One pipeline, parameterized by top_k

`run_variant(top_k)` scores the golden set through the pipeline and returns the per-question rows plus means. A and B are the *same function* with different arguments — that is what makes this a controlled A/B rather than two ad-hoc scripts. The judge and faithfulness metric are hand-rolled inline with the **same rubric and scale** the lab uses: `InlineJudge` wraps `ChatGroq` (JSON verdicts, one retry, `{"error": ...}` on failure, local BGE `embed`); `InlineFaithfulnessMetric.score` sends ONE judge call asking for `{"claims": [...], "supported": [bool, ...]}` and scores `supported / total`.


In [ ]:
# --------------------------------------------------------------------------
# 3. One pipeline, parameterized by top_k — judge + faithfulness inline
# --------------------------------------------------------------------------
def _strip_code_fence(text: str) -> str:
    """Remove a surrounding markdown code fence (```json ... ```)."""
    lines = text.strip().splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()


class InlineJudge:
    """LLM-as-judge over ChatGroq + local BGE embeddings (hand-rolled).

    Mirrors the shared LLMJudge contract the lab uses: ``judge()`` returns
    the parsed JSON dict (one retry on a JSON failure, then {"error": ...});
    ``embed()`` returns local BGE vectors.
    """

    def __init__(self, llm: ChatGroq, embedder):
        self.llm = llm
        self.embedder = embedder

    def judge(self, instruction: str, prompt: str) -> dict:
        last_error = ""
        for attempt in range(2):
            full = f"{instruction}\n\n{prompt}\n\nRespond with ONLY a valid JSON object."
            if attempt == 1:
                full += " Respond with ONLY valid JSON."
            try:
                text = self.llm.invoke(full).content
                return json.loads(_strip_code_fence(text))
            except (json.JSONDecodeError, ValueError) as exc:
                last_error = str(exc)
        return {"error": f"inline judge: could not parse JSON after 2 attempts: {last_error}"}

    def embed(self, texts: list[str]) -> list[list[float]]:
        return self.embedder.embed_documents(texts)


class InlineFaithfulnessMetric:
    """Faithfulness: fraction of the answer's claims supported by context.

    Same rubric as the shared FaithfulnessMetric: one judge call asking for
    ``{"claims": [...], "supported": [bool, ...]}``; score = supported /
    total claims (0.0 when no claims parse).
    """

    def __init__(self, judge):
        self.judge = judge

    def score(self, question: str, context: str, answer: str) -> float:
        instruction = (
            "You are a faithfulness judge. Break the answer into atomic "
            "claims, then mark each claim as supported (true) or not (false) "
            "by the context."
        )
        prompt = (
            f"Context:\n{context}\n\nAnswer:\n{answer}\n\n"
            'Return JSON: {"claims": ["..."], "supported": [true, false, ...]}'
        )
        result = self.judge.judge(instruction, prompt)
        if "error" in result:
            return 0.0
        claims, supported = self._normalize(result)
        n = min(len(claims), len(supported))
        if n == 0:
            return 0.0
        return sum(1 for flag in supported[:n] if flag) / n

    @staticmethod
    def _normalize(result: dict) -> tuple[list[str], list[bool]]:
        """Coerce the judge's JSON into parallel (claims, supported) lists.

        The judge is asked for ``{"claims": [...], "supported": [...]}`` but
        coder models routinely simplify to a singular ``{"claim": "...",
        "supported": true}``. Accept both: read ``claims``/``claim`` and
        ``supported`` as a list OR a single bool (a scalar bool is applied to
        every claim).
        """
        claims = result.get("claims")
        if not claims:
            claim = result.get("claim")
            claims = [claim] if claim else []
        supported = result.get("supported")
        if isinstance(supported, bool):
            supported = [supported] * len(claims)
        elif not isinstance(supported, list):
            supported = []
        return list(claims), list(supported)


def build_indexes() -> dict[str, object]:
    """Per-doc index: {doc_key: retriever over that invoice's pages}."""
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": "cpu"},  # leave the shared GPU alone
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    retrievers: dict[str, object] = {}
    for doc_key in GOLDEN_DOCS:
        pdf_path = INVOICE_DIR / f"{doc_key}.pdf"
        pages = PyPDFLoader(str(pdf_path)).load()
        texts = [p.page_content for p in pages]
        vectors = embedder.embed_documents(texts)
        chunks = [
            Document(page_content=t, metadata={"doc": doc_key, "page": i})
            for i, t in enumerate(texts)
        ]
        store = FAISS.from_documents(
            chunks, embedding=_PrecomputedEmbeddings(texts, vectors, embedder)
        )
        retrievers[doc_key] = store.as_retriever(search_kwargs={"k": 20})
    return retrievers


def run_variant(retrievers: dict[str, object], top_k: int,
                llm, judge, faithfulness) -> dict[str, float]:
    """Run the golden set through the pipeline with a given ``top_k``.

    Returns mean scores per metric over the golden questions plus the
    per-question rows (doc, question, contained) for eyeballing.
    """
    per_metric: dict[str, list[float]] = {"faithfulness": [], "contained": [],
                                          "cosine": []}
    rows: list[dict] = []
    for doc_key in GOLDEN_DOCS:
        retriever = retrievers[doc_key]
        for question, reference in GOLDEN_QA[doc_key]:
            docs = retriever.invoke(question)[:top_k]
            context = "\n\n".join(d.page_content for d in docs)
            answer = llm.invoke(
                f"Context:\n{context}\n\nQuestion: {question}\n\n"
                "Answer concisely, quoting the exact figure or value from "
                "the context:"
            ).content.strip()
            per_metric["faithfulness"].append(
                faithfulness.score(question, context, answer))
            contained = reference_contained(answer, reference)
            per_metric["contained"].append(contained)
            per_metric["cosine"].append(
                cosine_similarity(judge.embed([answer])[0],
                                  judge.embed([reference])[0]))
            rows.append({"doc": doc_key, "question": question,
                         "contained": contained})
    means = {k: sum(v) / len(v) for k, v in per_metric.items()}
    means["rows"] = rows
    return means


## 4. Experiment — build once, run both variants

The index is built **once** and shared by both variants — only the retriever's `top_k` differs. Then both variants run over the golden set and each is scored on faithfulness, contained, and cosine. The thread cap below keeps the BLAS/OpenMP footprint small while several track agents share this box.


In [ ]:
# --------------------------------------------------------------------------
# 4. Experiment — build once, run both variants
# --------------------------------------------------------------------------
# Several track agents share this machine — cap BLAS/OpenMP threads so the
# BGE embedding step stays light on CPU and memory.
os.environ["OMP_NUM_THREADS"] = "2"
import torch  # noqa: E402
torch.set_num_threads(2)


class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call. embed_query is delegated to the real embedder so
    the store's retriever can embed queries.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]],
                 query_embedder: Embeddings):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))
        self._query_embedder = query_embedder

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._query_embedder.embed_query(text)


def run_experiment() -> dict:
    t0 = time.perf_counter()
    retrievers = build_indexes()
    index_s = time.perf_counter() - t0

    llm = ChatGroq(model=GROQ_MODEL_NAME, temperature=0.0)
    judge = InlineJudge(llm, HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    ))
    faithfulness = InlineFaithfulnessMetric(judge)

    t0 = time.perf_counter()
    variant_a = run_variant(retrievers, VARIANT_A_TOP_K, llm, judge, faithfulness)
    variant_b = run_variant(retrievers, VARIANT_B_TOP_K, llm, judge, faithfulness)
    eval_s = time.perf_counter() - t0

    return {
        "docs": GOLDEN_DOCS,
        "n_questions": sum(len(GOLDEN_QA[d]) for d in GOLDEN_DOCS),
        "index_s": index_s,
        "eval_s": eval_s,
        "variant_a": variant_a,
        "variant_b": variant_b,
    }


## 5. Demo — print the artifact

The demo prints the A/B score table, the per-question containment rows, and the regression verdicts. Expect a subtlety: on faithfulness, B sometimes *beats* A — a single chunk is claim-support *density*, and a model that asserts less has less to contradict. The reference-anchored metrics (contained, cosine) gate strictly; faithfulness gets the tolerance band.


In [ ]:
# --------------------------------------------------------------------------
# 5. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 07-04 — Golden-set regression (evaluation/golden.py)")
    print(f"docs {exp['docs']}, {exp['n_questions']} golden questions")
    print("=" * 66)

    a, b = exp["variant_a"], exp["variant_b"]
    print(f"\n[1] Golden-set scores (mean over {exp['n_questions']} questions):")
    print(f"    {'metric':<14} {'A (top_k=3)':>12} {'B (top_k=1)':>12}  delta")
    for key in ("faithfulness", "contained", "cosine"):
        print(f"    {key:<14} {a[key]:>12.3f} {b[key]:>12.3f}  "
              f"{a[key] - b[key]:+.3f}")

    print(f"\n[2] Per-question reference containment (eyeball the rows):")
    print(f"    {'doc':<14} {'A':>4} {'B':>4}  question")
    for ra, rb in zip(a["rows"], b["rows"]):
        print(f"    {ra['doc']:<14} {ra['contained']:>4.0f} "
              f"{rb['contained']:>4.0f}  {ra['question'][:52]}")

    print(f"\n[3] Regression gate (reference-anchored strict, faithfulness "
          f"tolerated):")
    a_f, b_f = a["faithfulness"], b["faithfulness"]
    for key in ("contained", "cosine"):
        status = "PASS" if a[key] >= b[key] else "FAIL"
        print(f"    [{status}] A >= B for {key}")
    status = "PASS" if a_f >= b_f - FAITHFULNESS_TOLERANCE else "FAIL"
    print(f"    [{status}] A >= B - {FAITHFULNESS_TOLERANCE:.2f} for faithfulness")

    print(f"\n[4] Timing: index {exp['index_s']:.1f}s, evaluate both "
          f"{exp['eval_s']:.1f}s")

    print(f"\n[5] Takeaway")
    print("    A golden set is the regression contract: the same questions,")
    print("    the same references, forever. Variant B is a realistic silent")
    print("    regression — one context chunk instead of three — and the gate")
    print("    is 'reference pipeline >= new pipeline'. Two subtleties the")
    print("    numbers teach: (1) faithfulness is claim-support DENSITY, not")
    print("    completeness — a single chunk can score higher because the")
    print("    model asserts less; judge metrics get a tolerance. (2) the")
    print("    reference-anchored metrics gate strictly — when a change")
    print("    passes unit tests but drops a golden containment/cosine score,")
    print("    this gate stops it from shipping. Always eyeball the rows.")


## 6. Verification gate

The gate enforces the regression contract: `A >= B` for contained and cosine, `A >= B - 0.15` for faithfulness, a floor that the golden set is actually answerable, and all scores in [0, 1]. A change that passes unit tests but drops a golden score is stopped here. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 6. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    a, b = exp["variant_a"], exp["variant_b"]

    checks.append((f"{exp['n_questions']} golden questions evaluated (>= 6)",
                   exp["n_questions"] >= 6))

    for key in ("faithfulness", "contained", "cosine"):
        checks.append((f"A {key} in [0, 1]", 0.0 <= a[key] <= 1.0))
        checks.append((f"B {key} in [0, 1]", 0.0 <= b[key] <= 1.0))

    # The regression contract: A beats B on every metric. Reference-anchored
    # metrics gate strictly; the judge-based faithfulness metric gets the
    # tolerance band (claim-support density is not monotonic in context and
    # the judge is noisy on a 10-question sample).
    for key in ("contained", "cosine"):
        checks.append((f"regression gate: A >= B for {key}", a[key] >= b[key]))
    checks.append((
        f"regression gate: A >= B - {FAITHFULNESS_TOLERANCE} for faithfulness",
        a["faithfulness"] >= b["faithfulness"] - FAITHFULNESS_TOLERANCE))

    # A floor so the golden set is doing real work (not all-zeros).
    checks.append(("reference pipeline finds some answers (A contained > 0)",
                   a["contained"] > 0.0))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Indexing two small PDFs is fast; the 16 judge-scored generations (8 per variant) take a couple of minutes. `exp` holds both variants plus timing.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The A/B score table, the per-question containment rows, and the regression verdicts. Watch the faithfulness row — B can beat A there, which is exactly why it gets a tolerance band.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the invoice PDFs are intact and `GROQ_API_KEY` is set.


In [ ]:
verify_gate(exp)
